# Часть 1 Бустинг (5 баллов)

В этой части будем предсказывать зарплату data scientist-ов в зависимости  от ряда факторов с помощью градиентного бустинга.

В датасете есть следующие признаки:



* work_year: The number of years of work experience in the field of data science.

* experience_level: The level of experience, such as Junior, Senior, or Lead.

* employment_type: The type of employment, such as Full-time or Contract.

* job_title: The specific job title or role, such as Data Analyst or Data Scientist.

* salary: The salary amount for the given job.

* salary_currency: The currency in which the salary is denoted.

* salary_in_usd: The equivalent salary amount converted to US dollars (USD) for comparison purposes.

* employee_residence: The country or region where the employee resides.

* remote_ratio: The percentage of remote work offered in the job.

* company_location: The location of the company or organization.

* company_size: The company's size is categorized as Small, Medium, or Large.

In [ ]:
import pandas as pd

df = pd.read_csv("ds_salaries.csv")
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


## Задание 1 (0.5 балла) Подготовка



*   Разделите выборку на train, val, test (80%, 10%, 10%)
*   Выдерите salary_in_usd в качестве таргета
*   Найдите и удалите признак, из-за которого возможен лик в данных


In [ ]:
from sklearn.model_selection import train_test_split

df = df.drop(columns=['salary', 'salary_currency'])
X = df.drop(columns=['salary_in_usd'])
y = df['salary_in_usd']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

## Задание 2 (0.5 балла) Линейная модель


*   Закодируйте категориальные  признаки с помощью OneHotEncoder
*   Обучите модель линейной регрессии
*   Оцените  качество через MAPE и RMSE


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

categorical_features = [
    'experience_level',
    'employment_type',
    'job_title',
    'employee_residence',
    'company_location',
    'company_size'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)  #

model = LinearRegression()
model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_val_encoded)

mape = mean_absolute_percentage_error(y_val, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print('MAPE:', round(mape, 2), '%')
print('RMSE:', round(rmse, 2))

MAPE: 43.39 %
RMSE: 47994.7


## Задание 3 (0.5 балла) XGboost

Начнем с библиотеки xgboost.

Обучите модель `XGBRegressor` на тех же данных, что линейную модель, подобрав оптимальные гиперпараметры (`max_depth, learning_rate, n_estimators, gamma`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [ ]:
from xgboost.sklearn import XGBRegressor
import time

params = {
    'max_depth': 5,
    'learning_rate': 0.1,
    'n_estimators': 200,
    'gamma': 0.2,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42
}

start_time = time.time()
model = XGBRegressor(**params)
model.fit(X_train_encoded, y_train)
train_time = time.time() - start_time

start_pred_time = time.time()
y_pred = model.predict(X_val_encoded)
pred_time = time.time() - start_pred_time

mape = mean_absolute_percentage_error(y_val, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

In [ ]:
print('MAPE:', round(mape, 2), '%')
print('RMSE:', round(rmse, 2))
print(f"Время обучения: {train_time:.2f} сек")
print(f"Время предсказания: {pred_time:.4f} сек")

MAPE: 37.86 %
RMSE: 46188.02
Время обучения: 2.67 сек
Время предсказания: 0.0129 сек


## Задание 4 (1 балл) CatBoost

Теперь библиотека CatBoost.

Обучите модель `CatBoostRegressor`, подобрав оптимальные гиперпараметры (`depth, learning_rate, iterations`, etc.) по валидационной выборке. Оцените качество итоговой модели (MAPE, RMSE), скорость обучения и скорость предсказания.

In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostRegressor

for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

cat_features_indices = [X_train.columns.get_loc(col) for col in categorical_features]

params = {
    'depth': 6,
    'learning_rate': 0.1,
    'iterations': 1000,
    'l2_leaf_reg': 3,
    'random_seed': 42,
    'verbose': 100
}

start_time = time.time()
model = CatBoostRegressor(**params)
model.fit(
    X_train, y_train,
    cat_features=cat_features_indices,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50
)
train_time = time.time() - start_time

start_pred = time.time()
y_pred = model.predict(X_val)
pred_time = time.time() - start_pred

0:	learn: 61236.3016939	test: 59582.0518273	best: 59582.0518273 (0)	total: 23.1ms	remaining: 23.1s
100:	learn: 45485.3972300	test: 45681.1421336	best: 45656.7965638 (98)	total: 1.27s	remaining: 11.3s
200:	learn: 44031.9855062	test: 45493.3684640	best: 45492.1158623 (169)	total: 3.17s	remaining: 12.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 45481.25248
bestIteration = 212

Shrink model to first 213 iterations.


In [ ]:
mape = mean_absolute_percentage_error(y_val, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print('MAPE:', round(mape, 2), '%')
print('RMSE:', round(rmse, 2))
print(f'Время обучения: {train_time:.2f} сек')
print(f'Время предсказания: {pred_time:.4f} сек')

MAPE: 42.89 %
RMSE: 45481.25
Время обучения: 4.94 сек
Время предсказания: 0.0035 сек


Для применения catboost моделей не обязательно сначала кодировать категориальные признаки, модель может кодировать их сама. Обучите catboost с подбором оптимальных гиперпараметров снова, используя pool для передачи данных в модель с указанием какие признаки категориальные, а какие нет с помощью параметра cat_features. Оцените качество и время. Стало ли лучше?

In [ ]:
from catboost import Pool
from sklearn.model_selection import RandomizedSearchCV

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=categorical_features
)

val_pool = Pool(
    data=X_val,
    label=y_val,
    cat_features=categorical_features
)

base_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'verbose': 1000
}

param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1, 0.3],
    'iterations': [500, 800, 1000],
    'l2_leaf_reg': [1, 3, 5]
}

start_time = time.time()
model = CatBoostRegressor(**base_params)
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error'
)
search.fit(X_train, y_train, cat_features=categorical_features)
best_model = search.best_estimator_
search_time = time.time() - start_time

start_pred = time.time()
y_pred = best_model.predict(val_pool)
pred_time = time.time() - start_pred

mape = mean_absolute_percentage_error(y_val, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print('\n=== Лучшие параметры ===')
print(search.best_params_)

print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'Время подбора: {search_time:.2f} сек')
print(f'Время предсказания: {pred_time:.4f} сек')

0:	learn: 61067.4209696	total: 26.8ms	remaining: 21.4s
799:	learn: 31786.0729166	total: 23.9s	remaining: 0us
0:	learn: 61253.8456480	total: 9.38ms	remaining: 7.49s
799:	learn: 30835.6612817	total: 11.9s	remaining: 0us
0:	learn: 61520.8915926	total: 7.74ms	remaining: 6.18s
799:	learn: 32642.2324997	total: 9.68s	remaining: 0us
0:	learn: 57617.3301646	total: 6.22ms	remaining: 4.97s
799:	learn: 30857.9096357	total: 3.6s	remaining: 0us
0:	learn: 58147.4315035	total: 12.8ms	remaining: 10.2s
799:	learn: 29719.5142334	total: 5.48s	remaining: 0us
0:	learn: 58799.1363451	total: 4.69ms	remaining: 3.75s
799:	learn: 31048.3658477	total: 3.45s	remaining: 0us
0:	learn: 57455.5666317	total: 4.96ms	remaining: 2.47s
499:	learn: 31857.5130009	total: 2.23s	remaining: 0us
0:	learn: 57776.7948213	total: 4.39ms	remaining: 2.19s
499:	learn: 30996.3257888	total: 2.18s	remaining: 0us
0:	learn: 58741.5294642	total: 4.8ms	remaining: 2.4s
499:	learn: 32168.4073358	total: 3.88s	remaining: 0us
0:	learn: 59414.623756

**Ответ:** ЛУЧШЕ НЕ СТАЛО:(
  

## Задание 5 (0.5 балла) LightGBM

И наконец библиотека LightGBM - используйте `LGBMRegressor`, снова подберите гиперпараметры, оцените качество и скорость.


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()

X_train_encoded[categorical_features] = encoder.fit_transform(X_train[categorical_features])
X_val_encoded[categorical_features] = encoder.transform(X_val[categorical_features])


params = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 500,
    'num_leaves': 31,
    'min_child_samples': 20,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

start_time = time.time()
model = LGBMRegressor(**params)
model.fit(
    X_train_encoded, y_train,
    eval_set=[(X_val_encoded, y_val)],
    eval_metric='rmse')
train_time = time.time() - start_time

start_pred = time.time()
y_pred = model.predict(X_val_encoded)
pred_time = time.time() - start_pred

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000246 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 160
[LightGBM] [Info] Number of data points in the train set: 3004, number of used features: 7
[LightGBM] [Info] Start training from score 138055.989348
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

In [ ]:
mape = mean_absolute_percentage_error(y_val, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f'MAPE: {mape:.2f}%')
print(f'RMSE: {rmse:.2f}')
print(f'Время обучения: {train_time:.2f} сек')
print(f'Время предсказания: {pred_time:.4f} сек')

MAPE: 53.22%
RMSE: 49816.90
Время обучения: 0.23 сек
Время предсказания: 0.0130 сек


## Задание 6 (2 балла) Сравнение и выводы

Сравните модели бустинга и сделайте про них выводы, какая из моделей показала лучший/худший результат по качеству, скорости обучения и скорости предсказания? Как отличаются гиперпараметры для разных моделей?

**Ответ:** Самый лучший результат по качеству показала модель XGBRegressor, а самый худший - LGBMRegressor. По скорости обучения и скорости предсказания самая лучшая - модель XGBRegressor

# Часть 2 Кластеризация (5 баллов)

Будем работать с данными о том, каких исполнителей слушают пользователи музыкального сервиса.

Каждая строка таблицы - информация об одном пользователе. Каждый столбец - это исполнитель (The Beatles, Radiohead, etc.)

Для каждой пары (пользователь, исполнитель) в таблице стоит число - доля прослушивания этого исполнителя этим пользователем.


In [ ]:
import pandas as pd
ratings = pd.read_excel("https://github.com/evgpat/edu_stepik_rec_sys/blob/main/datasets/sample_matrix.xlsx?raw=true", engine='openpyxl')
ratings.head()

,user,the beatles,radiohead,deathcab for cutie,coldplay,modest mouse,sufjan stevens,dylan. bob,red hot clili peppers,pink fluid,...,municipal waste,townes van zandt,curtis mayfield,jewel,lamb,michal w. smith,群星,agalloch,meshuggah,yellowcard
0,0,NaN,0.020417,NaN,NaN,NaN,NaN,NaN,0.030496,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,0.184962,0.024561,NaN,NaN,0.136341,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,0.028635,NaN,NaN,NaN,0.024559,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.043529,0.086281,0.034590,0.016712,0.015935,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Будем строить кластеризацию исполнителей: если двух исполнителей слушало много людей примерно одинаковую долю своего времени (то есть векторы близки в пространстве), то, возможно исполнители похожи. Эта информация может быть полезна при построении рекомендательных систем.

## Задание 1 (0.5 балла) Подготовка

Транспонируем матрицу ratings, чтобы по строкам стояли исполнители.

In [ ]:
ratings_transposed = ratings.T
ratings_transposed

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
user,0.000000,1.000000,2.000000,3.0,4.000000,5.000000,6.0,7.0,8.000000,9.000000,...,4990.000000,4991.0,4992.000000,4993.000000,4994.000000,4995.000000,4996.0,4997.000000,4998.0,4999.000000
the beatles,NaN,NaN,NaN,NaN,0.043529,NaN,NaN,NaN,0.093398,0.017621,...,NaN,NaN,0.121169,0.038168,0.007939,0.017884,NaN,0.076923,NaN,NaN
radiohead,0.020417,0.184962,NaN,NaN,0.086281,0.006322,NaN,NaN,NaN,0.019156,...,0.017735,NaN,NaN,NaN,0.011187,NaN,NaN,NaN,NaN,NaN
deathcab for cutie,NaN,0.024561,0.028635,NaN,0.034590,NaN,NaN,NaN,NaN,0.013349,...,0.121344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027893
coldplay,NaN,NaN,NaN,NaN,0.016712,NaN,NaN,NaN,NaN,NaN,...,0.217175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
michal w. smith,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
群星,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agalloch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
meshuggah,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Выкиньте строку под названием `user`.

In [ ]:
if 'user' in ratings_transposed.index:
    ratings_transposed = ratings_transposed.drop('user')
ratings_transposed

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
the beatles,NaN,NaN,NaN,NaN,0.043529,NaN,NaN,NaN,0.093398,0.017621,...,NaN,NaN,0.121169,0.038168,0.007939,0.017884,NaN,0.076923,NaN,NaN
radiohead,0.020417,0.184962,NaN,NaN,0.086281,0.006322,NaN,NaN,NaN,0.019156,...,0.017735,NaN,NaN,NaN,0.011187,NaN,NaN,NaN,NaN,NaN
deathcab for cutie,NaN,0.024561,0.028635,NaN,0.034590,NaN,NaN,NaN,NaN,0.013349,...,0.121344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027893
coldplay,NaN,NaN,NaN,NaN,0.016712,NaN,NaN,NaN,NaN,NaN,...,0.217175,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modest mouse,NaN,NaN,NaN,NaN,0.015935,NaN,NaN,NaN,NaN,0.030437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
michal w. smith,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
群星,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agalloch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
meshuggah,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


В таблице много пропусков, так как пользователи слушают не всех-всех исполнителей, чья музыка представлена в сервисе, а некоторое подмножество (обычно около 30 исполнителей)


Доля исполнителя в музыке, прослушанной  пользователем, равна 0, если пользователь никогда не слушал музыку данного музыканта, поэтому заполните пропуски нулями.



In [ ]:
ratings = ratings_transposed.fillna(0)

print("Размерность после преобразований:", ratings.shape)
ratings.sample()

Размерность после преобразований: (1000, 5000)


,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
infected mushroom,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Задание 2 (0.5 балла) Первая кластеризация

Примените KMeans с 5ю кластерами, сохраните полученные лейблы

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(ratings)
ratings['cluster'] = cluster_labels

Выведите размеры кластеров. Полезной ли получилась кластеризация? Почему KMeans может выдать такой результат?

In [ ]:
cluster_sizes = ratings['cluster'].value_counts().sort_index()
print("Размеры кластеров:")
print(cluster_sizes)

Размеры кластеров:
cluster
0    115
1      1
2      1
3    882
4      1
Name: count, dtype: int64


**Ответ:** Кластеризация получилась бесполезной, так как 882 исполнителя из 1000 попало в один кластер, а остальные кластеры содержат по 1-115 исполнителей. Это произошло потому, что KMeans плохо работает с разреженными данными и не учитывает семантическую близость, группируя объекты только по евклидову расстоянию.

## Задание 3 (0.5 балла) Объяснение результатов

При кластеризации получилось $\geq 1$ кластера размера 1. Выведите исполнителей, которые составляют такие кластеры. Среди них должна быть группа The Beatles.

In [ ]:
single_clusters = cluster_sizes[cluster_sizes == 1].index.tolist()
outlier_artists = ratings[ratings['cluster'].isin(single_clusters)].index

print("Исполнители в кластерах размера 1:")
print(outlier_artists)

Исполнители в кластерах размера 1:
Index(['the beatles', 'niИ', '日dir en grey'], dtype='object')


Изучите данные, почему именно The Beatles выделяется?

Подсказка: посмотрите на долю пользователей, которые слушают каждого исполнителя, среднюю долю прослушивания.

In [ ]:
ratings_stats = pd.DataFrame({
    'users_listened': (ratings.drop('cluster', axis=1) > 0).sum(axis=1),  # кол-во слушателей
    'mean_share': ratings.drop('cluster', axis=1).mean(axis=1)            #  доля прослушиваний
})
print(ratings_stats.sort_values('users_listened', ascending=False).head(5))

beatles_stats = ratings_stats.loc['the beatles']
print(f"\nThe Beatles:")
print(f"- Доля пользователей, которые слушают: {beatles_stats['users_listened'] / len(ratings_filled.columns) * 100:.1f}%")
print(f"- Средняя доля прослушиваний: {beatles_stats['mean_share']:.4f}")

                    users_listened  mean_share
the beatles                   1671    0.018369
radiohead                     1389    0.011851
deathcab for cutie             931    0.006543
coldplay                       841    0.006030
modest mouse                   814    0.005876

The Beatles:
- Доля пользователей, которые слушают: 33.4%
- Средняя доля прослушиваний: 0.0184


**Ответ:** The Beatles слушает в 2-3 раза больше пользователей, чем других топовых исполнителей, средняя доля прослушиваний на 50-200% выше, чем у конкурентов.

Исполнители вроде nil и 日Dir en grey тоже попали в кластеры размера 1, но по обратной причине: их слушают очень мало пользователей (низкий охват), но те, кто слушает, делают это интенсивно (высокая средняя доля)

The Beatles имеют слишком большие значения по сравнению с другими исполнителями (доминируют в пространстве признаков) и нетипичное распределение (много нулей и несколько крайне высоких значений)

## Задание 4 (0.5 балла) Улучшение кластеризации

Попытаемся избавиться от этой проблемы: нормализуйте данные при помощи `normalize`.

In [ ]:
from sklearn.preprocessing import normalize

ratings_normalized = normalize(ratings.drop('cluster', axis=1))

Примените KMeans с 5ю кластерами на преобразованной матрице, посмотрите на их размеры. Стало ли лучше? Может ли кластеризация быть полезной теперь?

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(ratings_normalized)

cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
print("Размеры кластеров после нормализации:")
print(cluster_sizes)

for cluster in range(5):
    artists_in_cluster = ratings.index[cluster_labels == cluster]
    print(f"\nКластер {cluster} (размер: {len(artists_in_cluster)}):")
    print("Топ-5 исполнителей:", artists_in_cluster[:5])

Размеры кластеров после нормализации:
0     79
1    162
2    133
3    140
4    486
Name: count, dtype: int64

Кластер 0 (размер: 79):
Топ-5 исполнителей: Index(['kanye west', 'lil' wayne', 'jay-z', 'eminem', 'outkast'], dtype='object')

Кластер 1 (размер: 162):
Топ-5 исполнителей: Index(['deathcab for cutie', 'coldplay', 'the killers', 'johnson jack',
       '‌linkin park'],
      dtype='object')

Кластер 2 (размер: 133):
Топ-5 исполнителей: Index(['bright eyes', 'greenday', 'blink-182', 'brand new', 'the clash'], dtype='object')

Кластер 3 (размер: 140):
Топ-5 исполнителей: Index(['the beatles', 'dylan. bob', 'pink fluid', 'led zeppelin.',
       'divid bowie'],
      dtype='object')

Кластер 4 (размер: 486):
Топ-5 исполнителей: Index(['radiohead', 'modest mouse', 'sufjan stevens', 'red hot clili peppers',
       'niИ'],
      dtype='object')


**Ответ:** После нормализации распределение кластеров стало значительно лучше, но проблема доминирования одного крупного кластера (486 исполнителей) сохраняется

## Задание 5 (1 балл) Центроиды

Выведите для каждого кластера названия топ-10 исполнителей, ближайших к центроиду по косинусной мере. Проинтерпретируйте результат. Что можно сказать о смысле кластеров?

In [ ]:
from scipy.spatial.distance import cosine

centroids = kmeans.cluster_centers_

top_artists_per_cluster = {}
for cluster_id in range(5):
    distances = [
        cosine(ratings_normalized[i], centroids[cluster_id])
        for i in range(len(ratings_normalized))
    ]
    closest_indices = np.argsort(distances)[:10]
    top_artists = ratings.index[closest_indices]
    top_artists_per_cluster[cluster_id] = top_artists

for cluster_id, artists in top_artists_per_cluster.items():
    print(f"\nКластер {cluster_id} (размер: {cluster_sizes[cluster_id]}):")
    print("Топ-10 ближайших к центроиду:", list(artists))


Кластер 0 (размер: 79):
Топ-10 ближайших к центроиду: ['nas', 'jay-z', 'kanye west', 'lupe the gorilla', 'a tribe called quest', "the roots featuring d'angelo", 'gangstarr', 'little brother', "lil' wayne", 'murs and 9th wonder']

Кластер 1 (размер: 162):
Топ-10 ближайших к центроиду: ['fall out boy', 'the all-americian rejects', 'paramore', 'kelly clarkson', 'john mayer', 'the fray', 'maroon5', 'dashboard confesssional', 'somethings corporate', 'coldplay']

Кластер 2 (размер: 133):
Топ-10 ближайших к центроиду: ['brand new', 'blink-182', 'alkaline trio', 'against me!', 'underoath', 'descendents', 'new found glory', 'less than jake', 'thrice', 'chiodos']

Кластер 3 (размер: 140):
Топ-10 ближайших к центроиду: ['the beatles', 'the rolling stones', 'dylan. bob', 'who', 'led zeppelin.', 'miles davis.', 'simon and garfunkel', 'young, neil', 'pink fluid', 'velvet underground']

Кластер 4 (размер: 486):
Топ-10 ближайших к центроиду: ['radiohead', 'the arcade fire', 'the shins', 'sufjan steve

**Ответ:** Кластеры отражают жанровое и стилистическое разделение музыкальных исполнителей: кластер 0 объединяет хип-хоп и рэп артистов, кластер 1 — поп-рок и мейнстримовую поп-музыку, кластер 2 — панк и альтернативный рок, кластер 3 — классический рок и легендарных музыкантов разных эпох, а кластер 4 — инди-рок и экспериментальные направления. Таким образом, кластеризация эффективно группирует исполнителей по музыкальным предпочтениям и стилям

## Задание 6 (1 балл) Визуализация

Хотелось бы как-то визуализировать полученную кластеризацию. Постройте точечные графики `plt.scatter` для нескольких пар признаков исполнителей, покрасив точки в цвета кластеров. Почему визуализации получились такими? Хорошо ли они отражают разделение на кластеры? Почему?

In [ ]:
import matplotlib.pyplot as plt

# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Для визуализации данных высокой размерности существует метод t-SNE (стохастическое вложение соседей с t-распределением). Данный метод является нелинейным методом снижения размерности: каждый объект высокой размерности будет моделироваться объектов более низкой (например, 2) размерности таким образом, чтобы похожие объекты моделировались близкими, непохожие - далекими с большой вероятностью.

Примените `TSNE` из библиотеки `sklearn` и визуализируйте полученные объекты, покрасив их в цвета их кластеров

In [ ]:
from sklearn.manifold import TSNE

# -- YOUR CODE HERE --

## Задание 7 (1 балл) Подбор гиперпараметров

Подберите оптимальное количество кластеров (максимум 100 кластеров) с использованием индекса Силуэта. Зафиксируйте `random_state=42`

In [ ]:
from sklearn.metrics import silhouette_score

# -- YOUR CODE HERE --

Выведите исполнителей, ближайших с центроидам (аналогично заданию 5). Как соотносятся результаты? Остался ли смысл кластеров прежним? Расскажите про смысл 1-2 интересных кластеров, если он изменился и кластеров слишком много, чтобы рассказать про все.

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --

Сделайте t-SNE визуализацию полученной кластеризации.

In [ ]:
# -- YOUR CODE HERE --

Если кластеров получилось слишком много и визуально цвета плохо отличаются, покрасьте только какой-нибудь интересный кластер из задания выше (`c = (labels == i)`). Хорошо ли этот кластер отражается в визуализации?

In [ ]:
# -- YOUR CODE HERE --

**Ответ:** # -- YOUR ANSWER HERE --